In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

# plot
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

import random
import numpy as np

import math

In [ ]:
# def functions
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
# # read data
# train = pd.read_csv("ProjectTrainingData.csv")
# test = pd.read_csv("ProjectTestData.csv")

# train.columns

# print(min(test.hour))
# print(max(test.hour))

# print(train.shape)
# print(test.shape)

In [ ]:
sample = pd.read_csv('sample.csv')

# 1 1000000 Sample

In [ ]:
# take 1000000 samples
# sample = train.sample(1000000, random_state = 613)

print(sample.click.value_counts())
print(len(sample[sample.click == 1])/len(sample)) # 17%

# get date of sample
sample['date'] = round((sample.hour - 14100000)/100).astype('int')

# get hour of sample
sample.hour = sample.hour - 14100000 - sample.date * 100

print(sample.columns) # can treat hour and date as continuous variables, but now let's just make it categorical

for i in range(2, 25):
        sample.iloc[:, i] = sample.iloc[:, i].astype('category') # change to categorical variables
sample = sample.iloc[:, 1:] # delete id

for cat_var in sample.columns:
    print(cat_var, sample[cat_var].nunique())

print(sample.columns[[10, 11]])

sample.drop(['device_id', 'device_ip'], axis = 1, inplace = True) # so many categories, drop these two variables for now

In [ ]:
# not sure need to be the same or not?
# for i in range(0,23):
#     X_test = X_test[X_test[X_test.columns[i]].isin(X_train.iloc[:,i])]

# len(X_test)
# y_test = y_test[X_test.index]

In [ ]:
from sklearn.model_selection import train_test_split

# Split validation
X_train, X_test, y_train, y_test = train_test_split(
    sample.iloc[:, 1:], sample['click'], test_size = 0.4, random_state = 613, stratify = sample['click'])

In [ ]:
from sklearn import preprocessing
X_train_ordinal = X_train.values
X_test_ordinal = X_test.values
les = []

for i in range(X_train_ordinal.shape[1]):
    le = preprocessing.LabelEncoder()
    le.fit(sample.iloc[:,1:].iloc[:, i])
    les.append(le)
    X_train_ordinal[:, i] = le.transform(X_train_ordinal[:, i])
    X_test_ordinal[:, i] = le.transform(X_test_ordinal[:, i])

## 1.1 Tree

### 1.1.1 No Tuning

In [ ]:
# X_test and y_test are indeed validation data
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier

random.seed(613)
clf = tree.DecisionTreeClassifier()
clf = clf.fit(X_train_ordinal, y_train)

# predict 
y_pred = clf.predict(X_test_ordinal)

In [ ]:
import itertools
import sklearn
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, classification_report
from sklearn.metrics import log_loss
    
cnf_matrix = confusion_matrix(y_test, y_pred)
np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=['0', '1'],
                      title='Confusion matrix, without normalization')

# Plot normalized confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=['0', '1'], normalize=True,
                      title='Normalized confusion matrix')

plt.show()

# Accuracy
print('Accuracy (out-of-sample): %.2f' % accuracy_score(y_test, y_pred))

# F1 score
print('F1 score (out-of-sample): ', f1_score(y_test, y_pred, average='macro'))

# Kappa score
print('Kappa score (out-of-sample): ', cohen_kappa_score(y_test, y_pred))

# Build a text report showing the main classification metrics (out-of-sample performance)
print('Classification report: ',classification_report(y_test, y_pred, target_names=['0', '1']))

from sklearn.metrics import roc_curve, auc
false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, y_pred)
roc_auc = auc(false_positive_rate, true_positive_rate)
print('AUC: ', roc_auc)

y_pred_prob = clf.predict_proba(X_test_ordinal)

print('Log loss: ', log_loss(y_test, y_pred_prob))

In [ ]:
# naive score
log_loss_naive = log_loss(y_test, np.repeat(0.17, len(y_test)))
log_loss_naive # 0.4571626429765626

In [ ]:
plt.hist(y_pred)

### 1.1.2 Tune Max_depth for Trees

In [ ]:
# Tunning max_depth

max_depths = range(5, 20, 1)
test_results = []
accuracy = []

for max_depth in max_depths:
    clf_max_depth = DecisionTreeClassifier(max_depth=max_depth)
    clf_max_depth.fit(X_train_ordinal, y_train)
    
    max_depth_pred = clf_max_depth.predict_proba(X_test_ordinal)
    y_pred = clf_max_depth.predict(X_test_ordinal)
        
    score_log_loss = log_loss(y_test, max_depth_pred)
    accuracy.append(accuracy_score(y_test, y_pred))
    test_results.append(score_log_loss)

print(max_depths[test_results.index(min(test_results))])

In [ ]:
# Tunning max_depth and min_samples_leaf

from sklearn.model_selection import GridSearchCV #http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

np.random.seed(613)
gs_dt = GridSearchCV(estimator=DecisionTreeClassifier(random_state=613),
                  param_grid=[{'max_depth': range(10, 25, 3), 
                              'min_samples_leaf':range(250, 500, 50)
                               # ,'min_samples_split':[2,3,4,5]
                              }],
                  scoring='neg_log_loss',
                  cv=5,
                  n_jobs=4)

gs_dt = gs_dt.fit(X_train,y_train)
print(gs_dt.best_score_)
print(gs_dt.best_params_)

In [ ]:
clf_gs_1 = DecisionTreeClassifier(max_depth=16,
                              criterion='gini',
                              min_samples_leaf=350,
                              # min_samples_split=2,
                              random_state=613)

clf_gs_1 = clf_gs_1.fit(X_train_ordinal, y_train)
y_pred = clf_gs_1.predict(X_test_ordinal)
y_pred_prob = clf_gs_1.predict_proba(X_test_ordinal)

In [ ]:
import itertools
import sklearn
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, classification_report
from sklearn.metrics import log_loss
    
cnf_matrix = confusion_matrix(y_test, y_pred)
np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=['0', '1'],
                      title='Confusion matrix, without normalization')

# Plot normalized confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=['0', '1'], normalize=True,
                      title='Normalized confusion matrix')

plt.show()

# Accuracy
print('Accuracy (out-of-sample): %.2f' % accuracy_score(y_test, y_pred))

# F1 score
print('F1 score (out-of-sample): ', f1_score(y_test, y_pred, average='macro'))

# Kappa score
print('Kappa score (out-of-sample): ', cohen_kappa_score(y_test, y_pred))

# Build a text report showing the main classification metrics (out-of-sample performance)
print('Classification report: ',classification_report(y_test, y_pred, target_names=['0', '1']))

from sklearn.metrics import roc_curve, auc
false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, y_pred)
roc_auc = auc(false_positive_rate, true_positive_rate)
print('AUC: ', roc_auc)

print('Log loss: ', log_loss(y_test, y_pred_prob))

## 1.2 Random Forest

In [ ]:
# Tuning max_depth

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, classification_report

max_depths = range(20, 100, 20)
test_results = []
accuracy = []

for max_depth in max_depths:
    clf_max_depth = RandomForestClassifier(n_estimators = 25, max_depth = max_depth, max_features = 0.3)
    clf_max_depth.fit(X_train_ordinal, y_train)
    
    max_depth_pred = clf_max_depth.predict_proba(X_test_ordinal)
    y_pred = clf_max_depth.predict(X_test_ordinal)

    score_log_loss = log_loss(y_test, max_depth_pred)
    accuracy.append(accuracy_score(y_test, y_pred))
    test_results.append(score_log_loss)
    
print(max_depths[test_results.index(min(test_results))])

In [ ]:
# Tuning min_samples_split

min_samples_splits = range(50, 250, 50)
test_results = []
accuracy = []

for min_samples_split in min_samples_splits:
    clf_min_samples_split = RandomForestClassifier(n_estimators = 25, max_depth = 20, min_samples_split = min_samples_split, max_features = 0.3)
    clf_min_samples_split.fit(X_train_ordinal, y_train)
    
    min_samples_split_pred = clf_min_samples_split.predict_proba(X_test_ordinal)
    y_pred = clf_min_samples_split.predict(X_test_ordinal)

    score_log_loss = log_loss(y_test, min_samples_split_pred)
    accuracy.append(accuracy_score(y_test, y_pred))
    test_results.append(score_log_loss)
    
print(min_samples_splits[test_results.index(min(test_results))])

In [ ]:
# Tuning min_samples_leaf

min_samples_leafs = range(10, 125, 25)
test_results = []
accuracy = []

for min_samples_leaf in min_samples_leafs:
    clf_min_samples_leaf = RandomForestClassifier(n_estimators = 25, max_depth = 20, min_samples_split = 150, min_samples_leaf = min_samples_leaf, max_features = 0.3)
    clf_min_samples_leaf.fit(X_train_ordinal, y_train)
    
    min_samples_leaf_pred = clf_min_samples_leaf.predict_proba(X_test_ordinal)
    y_pred = clf_min_samples_leaf.predict(X_test_ordinal)

    score_log_loss = log_loss(y_test, min_samples_leaf_pred)
    accuracy.append(accuracy_score(y_test, y_pred))
    test_results.append(score_log_loss)
    
print(min_samples_leafs[test_results.index(min(test_results))])

In [ ]:
test_results

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import itertools
import sklearn
from sklearn.metrics import confusion_matrix

randomForest = RandomForestClassifier(max_depth = 20, n_estimators = 25, min_samples_split = 150, min_samples_leaf = 10, max_features=0.3)

np.random.seed(613)
randomForest.fit(X_train_ordinal, y_train)

# Feature Importance
names = sample.columns[1:]
print('RF feature importance: ', sorted(zip(map(lambda x: round(x, 4), randomForest.feature_importances_), names), 
             reverse=True))

y_pred = randomForest.predict(X_test_ordinal)
y_pred_prob = randomForest.predict_proba(X_test_ordinal)

cnf_matrix = confusion_matrix(y_test, y_pred)
np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=['0', '1'],
                      title='Confusion matrix, without normalization')

# Plot normalized confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=['0', '1'], normalize=True,
                      title='Normalized confusion matrix')

plt.show()

# Accuracy
print('Accuracy (out-of-sample): %.2f' % accuracy_score(y_test, y_pred))

# F1 score
print('F1 score (out-of-sample): ', f1_score(y_test, y_pred, average='macro'))

# Kappa score
print('Kappa score (out-of-sample): ', cohen_kappa_score(y_test, y_pred))

# Build a text report showing the main classification metrics (out-of-sample performance)
print('Classification report: ',classification_report(y_test, y_pred, target_names=['0', '1']))

from sklearn.metrics import roc_curve, auc
false_positive_rate, true_positive_rate, thresholds = roc_curve(y_test, y_pred)
roc_auc = auc(false_positive_rate, true_positive_rate)
print('AUC: ', roc_auc)

print('Log loss: ', log_loss(y_test, y_pred_prob))